# LambdaG Visualization — Carmen in libros regum vs. Comparison Candidates

This notebook generates the local token-level LambdaG visualizations for the hold-out work `X_cilr`.

The poem remains excluded from training and calibration. For each candidate in the shared test corpus, the visualization compares the complete masked *Carmen in libros regum* with up to 2000 tokens of the candidate and 20,000 tokens from the remaining test-corpus authors. The HTML files are saved locally in the poem-specific folder.

The purpose of these visualizations is diagnostic and local: they show how the LambdaG signal is distributed across the sentences and tokens of the poem, complementing the repeated global authorship-verification experiment.


## Stage 1: Setup

The general corpora and processed data are stored one directory above this notebook. HTML outputs remain in the current poem-specific directory.


In [ ]:
from pathlib import Path
import numpy as np

from sentence_corpora import TwoLevelCorpus
from lambdag.visualize import lambdaG_visualize

# ---------------------------------------------------------
# Shared files produced by the general elegiac pipeline
# ---------------------------------------------------------
GENERAL_DIR = Path("..").resolve()

TRAIN_CORPUS_PATH = GENERAL_DIR / "train_corpus.pkl"
TEST_CORPUS_PATH = GENERAL_DIR / "test_corpus.pkl"
CILR_CORPUS_PATH = GENERAL_DIR / "cilr_corpus.pkl"
DATA_PROCESSED_DIR = GENERAL_DIR / "data_processed"

# Local output directory: stays inside the CILR poem folder
OUTPUT_DIR = Path("html_cilr")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [
    TRAIN_CORPUS_PATH,
    TEST_CORPUS_PATH,
    CILR_CORPUS_PATH,
    DATA_PROCESSED_DIR,
]:
    if not path.exists():
        raise FileNotFoundError(f"Required path not found: {path}")

train_corpus = TwoLevelCorpus.from_pickle(str(TRAIN_CORPUS_PATH))
test_corpus = TwoLevelCorpus.from_pickle(str(TEST_CORPUS_PATH))
holdout_corpus = TwoLevelCorpus.from_pickle(str(CILR_CORPUS_PATH))

# Sanity check: the uncertain poem must remain outside train/test.
train_pairs = {(s.author, s.work) for s in train_corpus}
test_pairs = {(s.author, s.work) for s in test_corpus}
assert ("X", "cilr") not in train_pairs
assert ("X", "cilr") not in test_pairs

print(f"General analysis directory: {GENERAL_DIR}")
print(f"HTML output directory: {OUTPUT_DIR.resolve()}")
print(f"train_corpus: {len(train_corpus)} sentences")
print(f"test_corpus:  {len(test_corpus)} sentences")
print(f"holdout_corpus (X_cilr): {len(holdout_corpus)} sentences")
print(f"Authors in test_corpus: {sorted(test_corpus.get_unique_values('author'))}")


## Stage 2: Visualization per Candidate

For each candidate represented in the test corpus:

1. sample up to 2000 tokens from that candidate, balanced across available works;
2. sample 20,000 reference tokens from the other test-corpus authors;
3. use the complete masked *Carmen in libros regum* as the questioned text;
4. align the masked sentences with the unmasked `.quesplit.txt` version when available;
5. save masked and unmasked HTML visualizations in `html_cilr/`.

These local visualizations complement the calibrated global AV results; they do not constitute a separate calibration.


In [ ]:
# Get candidates represented in the shared test corpus
authors_with_works = sorted(test_corpus.get_unique_values("author"))
print(f"Found {len(authors_with_works)} candidates in the test corpus")

# Reproducible sampling
rng = np.random.default_rng(seed=42)

# Complete questioned text: Carmen in libros regum
q_sentences = [s.text.split() for s in holdout_corpus]
print(f"Questioned sentences (X_cilr): {len(q_sentences)}")

# Load the unmasked sentence version once, for colour-coding
unmasked_sents = None
holdout_work = holdout_corpus[0].metadata["work"] if len(holdout_corpus) > 0 else "cilr"

quesplit_files = list(
    DATA_PROCESSED_DIR.glob(f"*{holdout_work}_sentences.quesplit.txt")
)

if quesplit_files:
    quesplit_file = quesplit_files[0]
    with quesplit_file.open("r", encoding="utf-8") as f:
        unmasked_sents = [line.strip().split() for line in f if line.strip()]

    print(
        f"Unmasked sentences loaded from {quesplit_file.name}: "
        f"{len(unmasked_sents)}"
    )

    if len(q_sentences) != len(unmasked_sents):
        print(
            "WARNING: masked/unmasked sentence count mismatch "
            f"({len(q_sentences)} vs {len(unmasked_sents)}); "
            "unmasked visualization will be skipped."
        )
        unmasked_sents = None
else:
    print(
        "No unmasked quesplit file found for pattern: "
        f"*{holdout_work}_sentences.quesplit.txt"
    )

for author in authors_with_works:
    print(f"\n{'=' * 60}")
    print(f"Processing candidate: {author}")
    print(f"{'=' * 60}")

    # Candidate text: up to 2000 tokens, balanced across available works
    known_corpus = test_corpus.filter_by_level("author", author)
    known_total_tokens = sum(len(s.text.split()) for s in known_corpus)
    known_target_tokens = min(2000, known_total_tokens)

    if known_target_tokens == 0:
        print(f"Skipping {author}: no usable text in test corpus")
        continue

    known_sentences_raw, known_breakdown = known_corpus.sample_balanced(
        tokens=known_target_tokens,
        rng=rng,
    )
    known_sentences = [list(s.text.split()) for s in known_sentences_raw]

    print(
        f"Known sample: {sum(len(s) for s in known_sentences)} tokens "
        f"from {len(known_sentences)} sentences"
    )
    print(f"Known breakdown: {known_breakdown}")

    # Reference population: 20,000 tokens from all other test-corpus authors
    ref_corpus = TwoLevelCorpus(
        [s for s in test_corpus if s.author != author],
        levels=test_corpus.levels,
    )
    ref_total_tokens = sum(len(s.text.split()) for s in ref_corpus)
    ref_target_tokens = min(20000, ref_total_tokens)

    ref_sentences_raw, ref_breakdown = ref_corpus.sample_balanced(
        tokens=ref_target_tokens,
        rng=rng,
    )
    ref_sentences = [list(s.text.split()) for s in ref_sentences_raw]

    print(
        f"Reference sample: {sum(len(s) for s in ref_sentences)} tokens "
        f"from {len(ref_sentences)} sentences"
    )
    print(f"Reference breakdown: {ref_breakdown}")

    # --- Masked visualization ---
    masked_path = OUTPUT_DIR / f"lambdag_output_{author}_masked.html"

    result = lambdaG_visualize(
        q_text=q_sentences,
        k_text=known_sentences,
        ref_text=ref_sentences,
        N=8,
        r=30,
        output="html",
        scale="relative",
        negative=True,
        order_by="importance",
        rng=rng,
        print_path=str(masked_path),
    )

    print(f"Masked HTML saved to: {masked_path}")
    print(f"Table shape: {result['table'].shape}")
    print(f"Sentence count: {result['table']['sentence_id'].nunique()}")

    # --- Unmasked colour-coded visualization ---
    if unmasked_sents is not None:
        unmasked_path = OUTPUT_DIR / f"lambdag_output_{author}_unmasked.html"

        result_unmasked = lambdaG_visualize(
            q_text=q_sentences,
            k_text=known_sentences,
            ref_text=ref_sentences,
            N=8,
            r=30,
            output="html",
            scale="relative",
            negative=True,
            order_by="",
            unmasked_sentences=unmasked_sents,
            rng=rng,
            print_path=str(unmasked_path),
        )

        print(f"Unmasked HTML saved to: {unmasked_path}")
    else:
        result_unmasked = None
        print("Skipping unmasked visualization")

    # --- Summary statistics ---
    table = result["table"]
    sentence_table = table.drop_duplicates("sentence_id")

    print("\nSummary statistics:")
    print(f"  Sentences: {table['sentence_id'].nunique()}")
    print(f"  Tokens: {len(table)}")
    print(f"  Mean sentence λG: {sentence_table['sentence_lambdaG'].mean():.3f}")
    print(f"  Std sentence λG: {sentence_table['sentence_lambdaG'].std():.3f}")
    print(
        f"  Positive tokens: {(table['lambdaG'] > 0).sum()} "
        f"({(table['lambdaG'] > 0).mean() * 100:.1f}%)"
    )
    print(
        f"  Negative tokens: {(table['lambdaG'] < 0).sum()} "
        f"({(table['lambdaG'] < 0).mean() * 100:.1f}%)"
    )


## Stage 3: Optional combined diagnostic

The candidate-specific HTML files generated above are the outputs used for local interpretation. The following cell is retained only as an optional diagnostic and does not enter the attribution results.


In [ ]:
# OPTIONAL DIAGNOSTIC: inspect a balanced 2000-token sample
# from the complete test corpus. This cell does not run LambdaG and is
# not part of the attribution workflow.

rng_combined = np.random.default_rng(seed=42)

known_total_tokens = sum(len(s.text.split()) for s in test_corpus)
known_target_tokens = min(2000, known_total_tokens)

known_sentences_raw, known_breakdown = test_corpus.sample_balanced(
    tokens=known_target_tokens,
    rng=rng_combined,
)

print(
    f"Optional combined sample: "
    f"{sum(len(s.text.split()) for s in known_sentences_raw)} tokens "
    f"from {len(known_sentences_raw)} sentences"
)
print(f"Breakdown: {known_breakdown}")
